In [0]:
df = spark.read.table("group_project.bronze.fact_shift")


In [0]:
df.display( )
df.printSchema()

In [0]:
total_Count = df.count()
disnct_Count = df.distinct().count()
duplicate_counts = total_Count - disnct_Count
print(f"Total number of records: {total_Count}")
print(f"Distinct number of records: {disnct_Count}")
print(f"Number of duplicate records: {duplicate_counts}")
import pyspark.sql.functions as f
df.groupby("shift_id","people_id").count().filter(f.col("count")>1).show()
#as we can see there are 10 duplcates records in the data set


In [0]:
df.groupby("shift_id","people_id").count().filter(f.col("count")>1).show()
df = df.dropDuplicates(["shift_id","people_id"])
df.groupby("shift_id","people_id").count().filter(f.col("count")>1).show()

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from common_utils.logging import get_logger
from common_utils.generic_functions import rename_columns , casting , drop_columns
import json

In [0]:
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 

In [0]:
import json
from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.generic_functions import (
    rename_columns,
    casting,
    drop_columns
)

# =========================================
# 0. Logger
# =========================================

logger = get_logger("b2s_FactWork.ipynb")
logger.info("Session started")

# =========================================
# 1. CONFIG PATH
# =========================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Setting configurations")

# =========================================
# 2. Read Config
# =========================================

with open(config_path, "r") as file:
    config = json.load(file)

# =========================================
# 3. CONFIG Sections
# =========================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]
source_table = source_config["source_table"]
target_table = target_config["target_table"]

logger.info("Configuration completed")

# =========================================
# 4. Fact work Transformations
# =========================================

def factshift_transformations(df, config):

    # -------------------------------------
    # Trim and select columns
    # -------------------------------------

    df = df.select(
        F.col("shift_id"),
        F.col("people_id"),
        F.col("county_id"),
        F.col("scheduled_start"),
        F.col("scheduled_end"),
    )

    
    # -------------------------------------
    # Cast columns
    # -------------------------------------

    df = casting(
        df,
        config.get("cast_columns", {})
    )

    
    # -------------------------------------
    # Droping duplicates of the key columns
    # -------------------------------------

    df = df.dropDuplicates(config["keys"])
                           

    # -------------------------------------
    # Select final columns
    # -------------------------------------

    logger.info(
        "[%s] Selecting final Silver columns",
        source_name
    )

    df = df.select(*config["select_columns"])

    return df


# =========================================
# 5. Reading Bronze Table
# =========================================

logger.info("Reading from source table [%s]", source_table)

df = spark.read.table(source_table)


# =========================================
# 6. Run Transformations
# =========================================

cleaned_df = factshift_transformations(
    df,
    config
)

logger.info(
    "[%s] Fact shift transformation completed",
    source_name
)


# =========================================
# 7. Data Quality Checks
# =========================================

logger.info("[%s] Running data quality checks", source_name)

# Check null primary keys and county names
# Get composite keys from config

key_columns = config.get("keys",[])

if len(key_columns) != 2:
    raise ValueError("Expected two composite key columns in config")



# -------------------------------------
# Check null primary keys
# -------------------------------------

invalid_records = cleaned_df.filter(
    F.col(key_columns[0]).isNull()
    | F.col(key_columns[1]).isNull()
)

# -------------------------------------
# Check duplicate composite keys
# -------------------------------------


duplicates = (
    cleaned_df
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

# -------------------------------------
# Validate records
# -------------------------------------



if invalid_records.limit(1).count() > 0:
    raise ValueError("Null or invalid Fact shift records found")

if duplicates.limit(1).count() > 0:
    raise ValueError("Duplicate records found")

logger.info("Fact shift data quality checks passed")


# =========================================
# 8. Writing into Silver Layer
# =========================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS group_project.silver"
)

(
    cleaned_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

logger.info(
    "[%s] Successfully loaded into Silver: %s",
    source_name,
    target_table
)
# I didnt apply the SCD type 2 in this table as it is statc table